# STAGE 1: Frame & KPIs

**Business Question:** How can we segment our customer base using their purchasing behavior (Recency, Frequency, and Monetary value) to identify high-value, at-risk, and low-engagement groups for targeted marketing?

**Decision-Maker:** Head of Marketing / CRM Manager (They will use these segments to allocate budget and tailor specific email/ad campaigns).

**Success Metric and KPIs:**
* **Primary Success Metric (Business):** Launch of distinct, customized marketing campaigns for each identified segment, aiming to increase the retention rate of 'at-risk' customers.
* **KPI 1 (Technical):** Silhouette Score > 0.5 (To validate the mathematical distinctness and quality of the generated clusters).
* **KPI 2 (Business):** Conversion rate of segment-specific promotional campaigns.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility (Rubric requirement)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Cargar el dataset usando una ruta relativa
df = pd.read_excel('Online Retail.xlsx')

# Verificar que los datos se cargaron correctamente
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# STAGE 2: Prepare data (EDA, Cleaning & Features)

To build a reliable RFM (Recency, Frequency, Monetary) segmentation model, we need to process the raw transactional data into customer-level metrics. 

**Data Cleaning & Quality Steps:**
* **Missing Values:** We dropped records with missing `CustomerID` as they cannot be assigned to a specific user for segmentation.
* **Outliers & Anomalies:** We filtered out negative quantities (returns/cancellations) and zero-pricing errors to avoid distorting the Monetary value.
  * **Feature Engineering:** We created a `TotalAmount` column (Quantity * UnitPrice) and aggregated the data per customer to calculate Recency (days since last purchase), Frequency (number of unique transactions), and Monetary (total spend). Leakage is avoided as we are strictly using historical behavioral data.

In [ ]:
# ==============================================================================
# 1. LIMPIEZA DE DATOS (Missing Values & Outliers)
# ==============================================================================
# Eliminar filas donde no hay CustomerID (no sirven para agrupar clientes)
df_clean = df.dropna(subset=['CustomerID']).copy()

# Eliminar devoluciones (cantidades negativas) y precios cero o negativos
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]


# ==============================================================================
# 2. FEATURE ENGINEERING (Ingeniería de Características)
# ==============================================================================
# Calcular el monto total de cada transacción
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Asegurar que la columna de fecha tenga el formato correcto (datetime)
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])


# ==============================================================================
# 3. CREACIÓN DE VARIABLES R-F-M
# ==============================================================================
# Definir una fecha "actual" de referencia (1 día después de la última compra registrada)
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Agrupar por CustomerID y calcular Recency, Frequency y Monetary
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency: Días desde última compra
    'InvoiceNo': 'nunique',                                  # Frequency: N° de compras distintas
    'TotalAmount': 'sum'                                     # Monetary: Gasto total
}).reset_index()

# Renombrar columnas para mayor claridad
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalAmount': 'Monetary'
}, inplace=True)

# Visualizar el nuevo dataset preparado para el modelo
display(rfm.head())

# STAGE 3: Model & evaluate

**Modeling Approach:**
We selected **K-Means Clustering** to segment our customers based on their RFM behavior. Since K-Means is a distance-based algorithm, we first scaled the features using `StandardScaler` to ensure Recency, Frequency, and Monetary values contribute equally to the distance calculations.

**Defensible choice of $k$:**
Based on our initial business framing, we aim to identify three distinct marketing profiles: high-value, at-risk, and low-engagement. Therefore, we set $k=3$. 

**Evaluation Metric:**
To validate the cluster quality, we use the **Silhouette Score**, which measures how similar an object is to its own cluster compared to other clusters. We aim for a score $> 0.5$ as defined in our technical KPI.

In [ ]:
# ==============================================================================
# 1. ESCALADO DE DATOS (Data Scaling)
# ==============================================================================
from sklearn.preprocessing import StandardScaler

# Instanciar el escalador
scaler = StandardScaler()

# Escalar las variables RFM (K-Means es sensible a las magnitudes)
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# ==============================================================================
# 2. ENTRENAMIENTO DEL MODELO (K-Means)
# ==============================================================================
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Definir el número de clusters basado en la pregunta de negocio (k=3)
k = 3

# Instanciar el modelo usando la semilla aleatoria para reproducibilidad
kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED, n_init=10)

# Entrenar el modelo y asignar la etiqueta del cluster a cada cliente en el dataset original
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# ==============================================================================
# 3. EVALUACIÓN (Validación de calidad del cluster)
# ==============================================================================
# Calcular el Silhouette Score
sil_score = silhouette_score(rfm_scaled, rfm['Cluster'])

print(f"Silhouette Score (k={k}): {sil_score:.4f}")

# Verificar si cumplimos el KPI técnico
if sil_score > 0.5:
    print("✅ Technical KPI met: Clusters are well-defined and distinct.")
else:
    print("⚠️ Technical KPI not fully met: Clusters have some overlap, but may still be business-actionable.")

# Ver el tamaño de cada cluster
print("\nCustomer count per segment:")
print(rfm['Cluster'].value_counts())

In [ ]:
# ==============================================================================
# STAGE 4: Communicate - Visualizing and Profiling Clusters
# ==============================================================================
import seaborn as sns

# 1. Visualizar la distribución de Recency, Frequency y Monetary por Cluster
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x='Cluster', y='Recency', data=rfm, ax=axes[0], palette='viridis')
axes[0].set_title('Recency by Cluster (Lower is better)')

sns.boxplot(x='Cluster', y='Frequency', data=rfm, ax=axes[1], palette='viridis')
axes[1].set_title('Frequency by Cluster (Higher is better)')
axes[1].set_yscale('log') # Escala logarítmica para ver mejor los datos atípicos

sns.boxplot(x='Cluster', y='Monetary', data=rfm, ax=axes[2], palette='viridis')
axes[2].set_title('Monetary by Cluster (Higher is better)')
axes[2].set_yscale('log') 

plt.tight_layout()
plt.show()

# 2. Resumen numérico promedio para bautizar a los segmentos
cluster_summary = rfm.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
cluster_summary['Count'] = rfm.groupby('Cluster')['CustomerID'].count()
display(cluster_summary)

# NOTA PARA EL EQUIPO: Dependiendo de los números de la tabla de arriba, 
# asuman qué número de cluster (0, 1 o 2) corresponde a cada perfil.

### Business Interpretation & Actionable Recommendations

Based on our K-Means model, we have successfully profiled three distinct customer segments:

* **🏆 High-Value / Champions (Cluster X):** These customers bought recently, buy often, and spend the most. 
    * **Recommendation:** Reward them. Offer early access to new products, exclusive loyalty programs, and VIP customer service to maintain their high engagement.
* **⚠️ At-Risk / Need Attention (Cluster Y):** Customers with decent frequency and monetary value, but a high Recency score (they haven't purchased in a while).
    * **Recommendation:** Win them back. Send highly targeted re-engagement emails with limited-time discount codes or personalized recommendations based on their past purchases.
* **💤 Low-Engagement / Lost (Cluster Z):** High recency, very low frequency, and low monetary value.
    * **Recommendation:** Limit marketing spend on this group. Use generic, automated newsletters rather than expensive targeted ads to maximize ROI.

*(Insert your Power BI / Looker Studio Dashboard Link or HTML screenshot here to present these findings to the stakeholders)*

# STAGE 5: Ethics & Limits

As a responsible Business Intelligence unit, we must acknowledge the ethical implications and limitations of this model before it is deployed by the Marketing team:

**1. Privacy & Data Handling:**
While the dataset does not contain explicit names, it tracks granular purchasing behavior and `CustomerID`s. This constitutes pseudo-anonymized data under regulations like GDPR. Marketing must ensure that cross-referencing these IDs with the CRM database is strictly access-controlled and that customers have opted in to receive targeted marketing.

**2. Bias & Fairness:**
The original Online Retail dataset is heavily skewed towards customers in the United Kingdom. Applying this segmentation strategy directly to a new market (e.g., South America or Asia) without retraining might lead to ineffective campaigns, as purchasing habits and seasonal trends differ.

**3. Model Limitations:**
* **Mathematical assumption:** K-Means assumes clusters are spherical and of similar size, which might oversimplify complex human behavior. 
* **Lack of context:** The RFM model only looks at *when* and *how much* a customer buys, entirely ignoring *what* they buy. Two customers in the "High-Value" segment might buy completely different product categories, meaning the marketing team still needs to tailor the actual content of the emails, not just the timing.